# Figure 5 — Environment Survey

One panel per environment from the `g200 / β=1 / det=0.97 / perm_balanced` DI-fitness GA runs.
Each panel has three columns.

---

### Left — population scatter

Hexbin density of the full GA population in $(E_s[\mathcal{F}^*],\; \chi_\mathrm{twist})$ space.

- ★ = rank-1 sigma (best GA outcome by mean free energy)
- ○ = sigma with $\chi = 0$ (used as the untwisted baseline)

The x-axis is centred on each environment's data cloud with a fixed span shared across environments.
The y-axis ($\chi_\mathrm{twist}$) matches Figure 4.

---

### Middle — $\Delta\mathcal{F}$ heatmap

Per-state free energy gain from the best sigma's action relabelling, relative to the untwisted baseline.
The free energy is averaged over the set $G$ of evaluated goals:

$$\Delta\mathcal{F}(s) \;=\;
\frac{1}{|G|}\sum_{g \in G} \mathcal{F}^*_{\sigma^\star}(s,\,g)
\;-\;
\frac{1}{|G|}\sum_{g \in G} \mathcal{F}^*_\mathrm{base}(s,\,g)$$

Positive (dark) means the relabelling raised the Blahut–Arimoto free energy at that state.
The direction of the gradient across the grid should align with the hexbin x-axis.

Overlaid with the goal-marginalised policy quiver $\pi_G$.

---

### Right — $\Delta H[\pi_G]$ heatmap

Per-state Shannon entropy reduction in the goal-marginalised optimal policy:

$$\Delta H(s) \;=\;
H\!\bigl[\pi_G^\mathrm{base}(\cdot \mid s)\bigr]
\;-\;
H\!\bigl[\pi_G^{\sigma^\star}(\cdot \mid s)\bigr]$$

where

$$H[p] = -\sum_a p(a)\log_2 p(a) \quad \text{(bits)}$$

and the **goal-marginalised policy** is

$$\pi_G(a \mid s) \;=\; \frac{1}{|G|}\sum_{g \in G} \pi^*(a \mid s,\,g)$$

Positive (green) means goals agree more on the best action at that state under the best sigma —
the relabelling has reduced goal-to-goal uncertainty in the policy.
Negative (red) would mean the relabelling increased disagreement, which should be rare.

Overlaid with $\pi_G$ (the best-sigma version).

---

Colour scales are consistent across all seven environments:
ΔF uses sequential **PuBuGn**; ΔH uses diverging **RdYlGn** symmetric around 0.

In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import numpy.ma as nma
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.legend_handler import HandlerTuple
from mpl_toolkits.axes_grid1 import make_axes_locatable

# gridvis pilot: co-located support + installed packages, no gridFour on path.
NOTEBOOK_PARENT = Path.cwd()
FOUR_ROOMS_NB   = NOTEBOOK_PARENT / 'four_rooms'
for p in (NOTEBOOK_PARENT, FOUR_ROOMS_NB):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from figure_support import FIGURES_DIR, DI_CMAP, DI_VMIN, DI_VMAX
from ga_report_helpers import (
    DEFAULT_HEX_GRIDSIZE,
    PAPER_FREE_CHI_YLIM,
    PAPER_FREE_CHI_YTICKS,
    load_run_summary,
    load_sigma_aggregates,
    _normalize_xy_point,
    _draw_reference_markers,
    _reference_marker_legend_handle,
    _style_colorbar_for_export,
    load_shallow_report_module,
)
from figure_support import build_env
from gridvis.display_twist import _plot_heatmap_panel
from gridvis import display as disp
from gridbench.marginal_policy import find_sigma_hive_dir, load_goal_marginal_policy

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load the shallow-report module for axis labels and tick helpers
_module, _ = load_shallow_report_module()
LABEL_FREE     = _module._LABEL_FREE_MEAN    # $E[\mathcal{F}]$
LABEL_CHI      = _module._LABEL_CHI_TWIST    # $\chi_{\mathrm{twist}}$
LABEL_DI_MEAN  = _module._LABEL_DI_MEAN      # $E[\mathcal{I}_D]$

print('Setup OK')

In [ ]:
# Paper-scoped: this ALife *conference-paper* figure surveys only its own
# environments (four_rooms + wrap_grid), pinned explicitly below via
# RUN_OVERRIDES.  The former MULTI_ROOT auto-scan is retired: the store
# now also holds twist-home-vectors *scalability* runs (9x9 and larger)
# — a different paper, whose hive_sigma is excluded from desktop syncs, so
# surveying them here is both out of scope and unreproducible.  Those scaling
# figures belong to the extension paper, not this one.
env_runs: dict[str, Path] = {}

# ---------------------------------------------------------------------------
# Manual overrides — from alife_paper_config.py (single source of truth).
# Both four_rooms and wrap_grid are pinned so figures 1, 5 and 6 share the
# exact same runs (paper-refs.json).
# ---------------------------------------------------------------------------
from figure_support import RUN_DIR as FOUR_ROOMS_RUN_DIR, WRAP_GRID_RUN_DIR

RUN_OVERRIDES: dict[str, Path] = {
    'four_rooms': FOUR_ROOMS_RUN_DIR,
    'wrap_grid': WRAP_GRID_RUN_DIR,
}

for env_id, override_path in RUN_OVERRIDES.items():
    if override_path.exists() and any(override_path.glob('*.summary.json')):
        env_runs[env_id] = override_path
        print(f'  OVERRIDE {env_id} → {override_path.name}')
    else:
        print(f'  WARNING: override path not found for {env_id}: {override_path}')

print(f'\nEnvironments with runs ({len(env_runs)}):\n')
for env_id, run_dir in env_runs.items():
    s = load_run_summary(run_dir)
    print(f'  {env_id:20s}  best_mean_free={s["best_mean_free"]:.3f}  {run_dir.name}')


## Untwisted baseline

To isolate the effect of the action relabelling we compare against an **untwisted baseline**:
the Blahut–Arimoto optimal policy and free energy under the **standard (untwisted) MDP** with
the same environment dynamics ($\beta = 1$, $\det = 0.97$) but identity sigma ($\epsilon = 0$,
no action permutation).

For each environment the baseline is computed by solving:

$$\pi^\mathrm{base}(\cdot \mid s, g),\quad \mathcal{F}^\mathrm{base}(s, g)
\qquad \forall\, g \in G$$

with $\epsilon = 0$ so the twisted transition kernel reduces to the nominal kernel.

The baseline sigma is chosen as the population member with $\chi_\mathrm{twist} \approx 0$
(the member closest to the identity permutation); its hive directory is used as the cache
location.  Results are written to `baseline_marginal.npz` on first run.

In [ ]:
# ---------------------------------------------------------------------------
# Compute untwisted baseline marginal policy, entropy, mean free energy
# (epsilon=0, beta=1).  Cached to <chi0_hive_dir>/baseline_marginal.npz
# ---------------------------------------------------------------------------

from gridcore.info.decision_information import DecisionInformation
from gridcore.planning.state_distribution import UniformStateDistribution
from figure_support import _walls_for_env
from gridcore.envs import GridRoom

_BASELINE_KEYS = {'marginal_policy', 'entropy', 'mean_free', 'info', 'value'}


def _load_or_compute_baseline(
    run_dir: Path,
    chi0_hash: str,
    env_id: str,
    shape: tuple[int, int],
    determinism: float,
    beta: float = 1.0,
    theta: float = 1e-5,
    force: bool = False,
) -> dict:
    """Return dict with marginal_policy, entropy, mean_free for untwisted MDP.

    Cached to baseline_marginal.npz in the chi0 hive dir.
    Recomputes automatically if the cache is missing any required key.
    """
    hive_dir = find_sigma_hive_dir(run_dir, chi0_hash)
    if hive_dir is None:
        print(f'  [{env_id}] chi0 hive dir not found')
        return {}

    cache_path = hive_dir / 'baseline_marginal.npz'
    if cache_path.exists() and not force:
        cached = np.load(cache_path)
        if _BASELINE_KEYS.issubset(set(cached.files)):
            return {k: cached[k] for k in cached.files}
        print(f'  [{env_id}] cache stale (missing keys), recomputing … ', end='', flush=True)

    # Goal list from best sigma's metrics.npz
    best_hash   = load_run_summary(run_dir)['best_sigma_hash']
    best_hive   = find_sigma_hive_dir(run_dir, best_hash)
    goal_states = list(map(int, np.load(best_hive / 'metrics.npz')['goals']))

    if not cache_path.exists() or force:
        print(f'  [{env_id}] computing untwisted baseline — {len(goal_states)} goals … ', end='', flush=True)

    from gridcore.planning.policy import Policy
    policies, free_arrays, info_arrays, value_arrays = [], [], [], []
    for goal in goal_states:
        options = {'shape': shape, 'goals': [goal], 'manhattan': True, 'determinism': determinism}
        walls = _walls_for_env(env_id, shape)
        if walls is not None:
            options['walls'] = walls
        if env_id == 'wrap_grid':
            options['wrap'] = True

        env_g        = GridRoom(options)   # epsilon omitted → identity sigma
        state_dist   = UniformStateDistribution(env_g)
        di           = DecisionInformation(env_g, state_dist, theta=theta)
        policy, _, free_arr = di.get_opt_policy_Z_free_vector(beta=float(beta))
        info_arr     = di.get_decision_information_given_policy(policy)
        value_arr, _ = Policy.evaluate_policy(env_g, policy)
        policies.append(np.asarray(policy, dtype=float))
        free_arrays.append(np.asarray(free_arr, dtype=float))
        info_arrays.append(np.asarray(info_arr, dtype=float))
        value_arrays.append(np.asarray(value_arr, dtype=float))

    from gridbench.marginal_policy import compute_goal_marginal_policy
    marginal  = compute_goal_marginal_policy(np.stack(policies, axis=0))
    mean_free = np.stack(free_arrays, axis=0).mean(axis=0)
    mean_info = np.stack(info_arrays, axis=0).mean(axis=0)
    mean_value = np.stack(value_arrays, axis=0).mean(axis=0)

    p       = np.clip(marginal, 1e-15, None)
    entropy = -np.sum(p * np.log2(p), axis=1)
    entropy[marginal.sum(axis=1) < 1e-9] = 0.0

    np.savez(cache_path, marginal_policy=marginal, entropy=entropy, mean_free=mean_free, info=mean_info, value=mean_value)
    print('done — cached')
    return {'marginal_policy': marginal, 'entropy': entropy, 'mean_free': mean_free, 'info': mean_info, 'value': mean_value}


# Run for all environments (no-op if cache is fresh)
for env_id, run_dir in env_runs.items():
    agg       = load_sigma_aggregates(run_dir)
    agg       = agg[np.isfinite(agg['mean_free']) & np.isfinite(agg['chi_twist'])].copy()
    chi0_hash = str(agg.loc[agg['chi_twist'].idxmin(), 'sigma_hash'])
    shape     = next(
        tuple(int(v) for v in part.replace('shape=', '').split('x'))
        for part in run_dir.parts if part.startswith('shape=')
    )
    result = _load_or_compute_baseline(run_dir, chi0_hash, env_id, shape, determinism=0.97)
    if result:
        print(f'  {env_id}: mean_free max={result["mean_free"].max():.3f}')

print('\nBaseline ready.')

## Per-environment data loading and metric computation

For each environment the following quantities are loaded and derived.

### Goal-marginalised policy $\pi_G$

The best sigma's per-goal optimal policies $\{\pi^*(\cdot \mid s, g)\}_{g \in G}$ are loaded
from `hive_sigma/…/sigma_id=<hash>/goal-<g>.npz` and averaged uniformly over goals:

$$\pi_G(a \mid s) = \frac{1}{|G|} \sum_{g \in G} \pi^*(a \mid s,\, g)$$

### Mean free energy $\bar{\mathcal{F}}(s)$

The Blahut–Arimoto free energy $\mathcal{F}^*(s, g)$ is read from the same npz files and
averaged over goals:

$$\bar{\mathcal{F}}_{\sigma}(s) = \frac{1}{|G|} \sum_{g \in G} \mathcal{F}^*_\sigma(s,\, g)$$

### $\Delta\mathcal{F}(s)$ — free energy gain

Difference between the best sigma and the untwisted baseline, at each state:

$$\Delta\mathcal{F}(s) = \bar{\mathcal{F}}_{\sigma^\star}(s) - \bar{\mathcal{F}}_\mathrm{base}(s)$$

This is the per-state analogue of the x-axis displacement in the hexbin scatter.
States where the relabelling is most beneficial should show the largest $\Delta\mathcal{F}$.

### $\Delta H(s)$ — entropy reduction

Shannon entropy of the goal-marginalised policy, in bits:

$$H[\pi_G](s) = -\sum_a \pi_G(a \mid s)\,\log_2 \pi_G(a \mid s)$$

Entropy reduction induced by the relabelling:

$$\Delta H(s) = H[\pi_G^\mathrm{base}](s) - H[\pi_G^{\sigma^\star}](s)$$

High $\Delta H$ at a state means goals that previously disagreed on the best action now agree —
the relabelling has focused the goal-conditioned policies onto a common preferred direction.

In [ ]:
# ---------------------------------------------------------------------------
# Load aggregates, identify key sigmas, load per-env data, build env
# ---------------------------------------------------------------------------

def _shape_from_run(run_dir: Path) -> tuple[int, int]:
    for part in run_dir.parts:
        if part.startswith('shape='):
            h, w = part.replace('shape=', '').split('x')
            return int(h), int(w)
    raise ValueError(f'No shape= segment in {run_dir}')


def _load_mean_free(hive_dir: Path) -> np.ndarray | None:
    """Mean free energy per state, averaged over all evaluated goals."""
    arrays = []
    for gf in sorted(hive_dir.glob('goal-*.npz'), key=lambda p: int(p.stem.split('-')[1])):
        d = np.load(gf)
        if 'free' in d.files:
            arrays.append(np.asarray(d['free'], dtype=float))
    return np.stack(arrays, axis=0).mean(axis=0) if arrays else None


def _load_goal_arrays(hive_dir: Path, keys: list) -> dict:
    """Load per-goal arrays for the given keys and return their goal-averaged means.

    Returns a dict mapping each key to an ndarray (or None if no data found).
    Works for any combination of 'free', 'info', 'value', 'policy', etc.
    """
    buffers: dict = {k: [] for k in keys}
    for gf in sorted(hive_dir.glob('goal-*.npz'), key=lambda p: int(p.stem.split('-')[1])):
        d = np.load(gf)
        for k in keys:
            if k in d.files:
                buffers[k].append(np.asarray(d[k], dtype=float))
    return {
        k: np.stack(arrs, axis=0).mean(axis=0) if arrs else None
        for k, arrs in buffers.items()
    }


def _policy_entropy(marginal_policy: np.ndarray) -> np.ndarray:
    p   = np.clip(marginal_policy, 1e-15, None)
    ent = -np.sum(p * np.log2(p), axis=1)
    ent[marginal_policy.sum(axis=1) < 1e-9] = 0.0
    return ent


env_data: dict[str, dict] = {}

for env_id, run_dir in env_runs.items():
    summary  = load_run_summary(run_dir)
    agg      = load_sigma_aggregates(run_dir)
    agg      = agg[np.isfinite(agg['mean_free']) & np.isfinite(agg['chi_twist'])].copy()
    shape    = _shape_from_run(run_dir)

    best_hash = summary['best_sigma_hash']
    chi0_hash = str(agg.loc[agg['chi_twist'].idxmin(), 'sigma_hash'])
    best_row  = agg[agg['sigma_hash'] == best_hash].iloc[0]
    chi0_row  = agg[agg['sigma_hash'] == chi0_hash].iloc[0]

    best_hive = find_sigma_hive_dir(run_dir, best_hash)
    chi0_hive = find_sigma_hive_dir(run_dir, chi0_hash)

    # Per-state quantities for best sigma
    best_mean_free = _load_mean_free(best_hive) if best_hive else None
    best_marginal  = None
    if best_hive:
        best_marginal, _ = load_goal_marginal_policy(best_hive)

    # Absolute DI and value for best sigma (from pre-computed goal files)
    best_arrs       = _load_goal_arrays(best_hive, ['info', 'value']) if best_hive else {}
    best_mean_info  = best_arrs.get('info')
    best_mean_value = best_arrs.get('value')

    # Untwisted baseline (from cache)
    baseline = _load_or_compute_baseline(run_dir, chi0_hash, env_id, shape, determinism=0.97)
    baseline_mean_free = baseline.get('mean_free')
    baseline_entropy   = baseline.get('entropy')

    # Absolute DI and value for baseline sigma (from chi0 hive goal files)
    base_arrs           = _load_goal_arrays(chi0_hive, ['info', 'value']) if chi0_hive else {}
    baseline_mean_info  = baseline.get('info')
    baseline_mean_value = baseline.get('value')
    baseline_marginal_p = baseline.get('marginal_policy')

    # ΔF = F_best − F_baseline  (positive → relabelling increased free energy)
    delta_free = (
        best_mean_free - baseline_mean_free
        if best_mean_free is not None and baseline_mean_free is not None
        else None
    )

    # ΔI = I_best − I_baseline  (positive → relabelling increased DI cost)
    delta_info = (
        best_mean_info - baseline_mean_info
        if best_mean_info is not None and baseline_mean_info is not None
        else None
    )

    # ΔV = V_best − V_baseline  (positive → relabelling improved value)
    delta_value = (
        best_mean_value - baseline_mean_value
        if best_mean_value is not None and baseline_mean_value is not None
        else None
    )

    # Absolute entropies and ΔH
    best_entropy  = _policy_entropy(best_marginal) if best_marginal is not None else None
    base_entropy_arr = baseline_entropy  # already computed above from baseline cache
    delta_entropy = (
        base_entropy_arr - best_entropy
        if base_entropy_arr is not None and best_entropy is not None
        else None
    )

    env = build_env(env_id, shape=shape, goal=int(summary['goals'][0]), determinism=0.97)

    env_data[env_id] = dict(
        run_dir              = run_dir,
        agg                  = agg,
        best_hash            = best_hash,
        chi0_hash            = chi0_hash,
        best_raw_xy          = (float(best_row['mean_free']), float(best_row['chi_twist'])),
        chi0_raw_xy          = (float(chi0_row['mean_free']), float(chi0_row['chi_twist'])),
        shape                = shape,
        env                  = env,
        delta_free           = delta_free,
        marginal_policy      = best_marginal,
        delta_entropy        = delta_entropy,
        best_entropy         = best_entropy,
        baseline_entropy     = base_entropy_arr,
        best_mean_info       = best_mean_info,
        best_mean_value      = best_mean_value,
        baseline_marginal    = baseline_marginal_p,
        baseline_mean_info   = baseline_mean_info,
        baseline_mean_value  = baseline_mean_value,
        delta_info           = delta_info,
        delta_value          = delta_value,
    )
    df_str = f'ΔF max={delta_free.max():.3f}' if delta_free is not None else 'ΔF=None'
    dh_str = f'ΔH max={delta_entropy.max():.3f}' if delta_entropy is not None else 'ΔH=None'
    di_str = f'ΔI max={delta_info.max():.3f}' if delta_info is not None else 'ΔI=None'
    dv_str = f'ΔV max={delta_value.max():.3f}' if delta_value is not None else 'ΔV=None'
    print(f'{env_id}: ★ free={best_row["mean_free"]:.3f} chi={best_row["chi_twist"]:.3f}  {df_str}  {dh_str}  {di_str}  {dv_str}')


In [ ]:
# ---------------------------------------------------------------------------
# Axis limits and colour scales
# ---------------------------------------------------------------------------

HEX_GRIDSIZE = DEFAULT_HEX_GRIDSIZE
GLOBAL_YLIM  = PAPER_FREE_CHI_YLIM
CHI_YTICKS   = PAPER_FREE_CHI_YTICKS
y_span       = GLOBAL_YLIM[1] - GLOBAL_YLIM[0]

# X-axis: fixed span, per-env centred
per_env_spans = [
    d['agg']['mean_free'].max() - d['agg']['mean_free'].min()
    for d in env_data.values()
]
X_SPAN = float(max(per_env_spans)) + 1.0

for d in env_data.values():
    cx = float(d['agg']['mean_free'].mean())
    d['xlim'] = (cx - X_SPAN / 2, cx + X_SPAN / 2)

def _norm_env(x_raw, y_raw, xlim):
    xs = xlim[1] - xlim[0]
    return ((np.asarray(x_raw) - xlim[0]) / xs,
            (np.asarray(y_raw) - GLOBAL_YLIM[0]) / y_span)

# Pass 1: global max hexbin count
global_max_count = 0
for d in env_data.values():
    x_n, y_n = _norm_env(d['agg']['mean_free'].values, d['agg']['chi_twist'].values, d['xlim'])
    fig_tmp, ax_tmp = plt.subplots()
    hb_tmp = ax_tmp.hexbin(x_n, y_n, gridsize=HEX_GRIDSIZE, extent=[0.0, 1.0, 0.0, 1.0])
    global_max_count = max(global_max_count, int(np.max(hb_tmp.get_array())))
    plt.close(fig_tmp)

HEX_VMAX = float(np.log1p(global_max_count))
HEX_TICK_POS, HEX_TICK_LABELS = _module._log1p_count_tick_spec(float(global_max_count))

# ΔF colour scale: symmetric diverging centred at 0, spanning the actual data
# range across all environments so that colorbar ticks always match cell annotations.
delta_free_vals = [
    d['delta_free'][np.isfinite(d['delta_free'])]
    for d in env_data.values() if d['delta_free'] is not None
]
if delta_free_vals:
    all_df = np.concatenate(delta_free_vals)
    _df_abs_max = float(max(abs(float(all_df.min())), abs(float(all_df.max()))))
    DELTA_FREE_CLIM = (-_df_abs_max, _df_abs_max)
else:
    DELTA_FREE_CLIM = (-3.0, 3.0)

# ΔH colour scale: symmetric diverging
delta_ent_vals = [
    d['delta_entropy'][np.isfinite(d['delta_entropy']) & (np.abs(d['delta_entropy']) > 1e-9)]
    for d in env_data.values() if d['delta_entropy'] is not None
]
if delta_ent_vals:
    _abs_max = float(np.percentile(np.abs(np.concatenate(delta_ent_vals)), 98))
    DELTA_ENT_CLIM = (-_abs_max, _abs_max)
else:
    DELTA_ENT_CLIM = (-2.0, 2.0)

print(f'X span           : {X_SPAN:.2f}')
print(f'Hexbin max count : {global_max_count}')
print(f'ΔF clim          : [{DELTA_FREE_CLIM[0]:.3f}, {DELTA_FREE_CLIM[1]:.3f}]')
print(f'ΔH clim          : [{DELTA_ENT_CLIM[0]:.3f}, {DELTA_ENT_CLIM[1]:.3f}]')

In [ ]:
CELL_SIZE      = 0.38

# Truncated PRGn: sample 0.10–0.90 to limit saturation while keeping
# purple = positive ΔF, white = zero, green = negative ΔF (colorblind-friendly)
def _make_cmap_trunc(name='PRGn', lo=0.10, hi=0.90, n=256):
    import matplotlib as mpl
    from matplotlib.colors import LinearSegmentedColormap
    base = mpl.colormaps[name]
    return LinearSegmentedColormap.from_list(
        f'{name}_trunc', base(np.linspace(lo, hi, n)), N=n)

DELTA_FREE_CMAP = _make_cmap_trunc('PRGn', 0.10, 0.90)  # purple=+ΔF, white=0, green=−ΔF
DELTA_ENT_CMAP  = 'RdYlGn'   # diverging: red = entropy up, green = entropy down


def _apply_dashed_border(ax):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linestyle((0, (4, 2)))
        spine.set_linewidth(1.5)
        spine.set_color('#444444')
        spine.set_zorder(10)


def _annotate_heatmap(ax, data, env, *, cmap=None, clim=None, size=5, shift=0.35):
    """Annotate non-wall cells with 1-dp values.

    Text colour adapts to background luminance so annotations stay readable
    on both light and dark cells.  Pass *cmap* (name string or colormap object)
    and *clim* (vmin, vmax) to enable adaptation; omit for fixed dark text.
    """
    import matplotlib

    _cmap = (matplotlib.colormaps[cmap] if isinstance(cmap, str) else cmap) if cmap is not None else None
    _vmin = float(clim[0]) if clim is not None else 0.
    _vmax = float(clim[1]) if clim is not None else 1.
    _span = max(_vmax - _vmin, 1e-9)

    wall_set = set(getattr(env, 'walls_flat', []))
    w   = env.shape[1]
    mat = data.reshape(env.shape)
    for row in range(env.shape[0]):
        for col in range(env.shape[1]):
            if row * w + col in wall_set:
                continue
            val = float(mat[row, col])
            if _cmap is not None:
                t = float(np.clip((val - _vmin) / _span, 0., 1.))
                r, g, b, _ = _cmap(t)
                # Perceived luminance — ITU-R BT.709
                lum = 0.2126 * r + 0.7152 * g + 0.0722 * b
                txt_color = 'white' if lum < 0.45 else '#333333'
            else:
                txt_color = '#333333'
            ax.text(col + shift, row + 0.6 * shift, f'{val:.1f}',
                    ha='center', va='center', fontsize=size, color=txt_color)


def plot_env_panel(env_id: str, d: dict, out_base: Path) -> None:
    agg             = d['agg']
    env             = d['env']
    best_raw_xy     = d['best_raw_xy']
    chi0_raw_xy     = d['chi0_raw_xy']
    delta_free      = d['delta_free']
    marginal_policy = d['marginal_policy']
    delta_entropy   = d['delta_entropy']
    best_hash       = d['best_hash']
    run_id          = d['run_dir'].name
    shape           = d['shape']
    xlim            = d['xlim']
    x_span_env      = xlim[1] - xlim[0]

    ph_grid = shape[0] * CELL_SIZE
    pw_grid = shape[1] * CELL_SIZE
    ph_hex  = ph_grid
    pw_hex  = ph_hex

    pad_l, pad_r, pad_b, pad_t = 0.65, 0.15, 0.60, 0.40
    gap  = 0.55
    gap2 = 0.65

    fig_w = pad_l + pw_hex + gap + pw_grid + gap2 + pw_grid + pad_r + 0.8
    fig_h = pad_b + ph_hex + pad_t
    y0    = pad_b / fig_h

    x_hex = pad_l / fig_w
    x_df  = (pad_l + pw_hex + gap) / fig_w
    x_ent = (pad_l + pw_hex + gap + pw_grid + gap2) / fig_w

    fig    = plt.figure(figsize=(fig_w, fig_h))
    ax_hex = fig.add_axes([x_hex, y0, pw_hex  / fig_w, ph_hex  / fig_h])
    ax_df  = fig.add_axes([x_df,  y0, pw_grid / fig_w, ph_grid / fig_h])
    ax_ent = fig.add_axes([x_ent, y0, pw_grid / fig_w, ph_grid / fig_h])

    div_hex = make_axes_locatable(ax_hex)
    cax_hex = div_hex.append_axes('right', size='2%', pad=0.06)
    div_df  = make_axes_locatable(ax_df)
    cax_df  = div_df.append_axes('right', size='2%', pad=0.06)
    div_ent = make_axes_locatable(ax_ent)
    cax_ent = div_ent.append_axes('right', size='2%', pad=0.06)

    # ===== hexbin =====
    x_raw = agg['mean_free'].values.astype(float)
    y_raw = agg['chi_twist'].values.astype(float)
    x_n, y_n = _norm_env(x_raw, y_raw, xlim)

    hb = ax_hex.hexbin(
        x_n, y_n, gridsize=HEX_GRIDSIZE, mincnt=1,
        extent=[0.0, 1.0, 0.0, 1.0], cmap='Purples', linewidths=0.0, zorder=2,
    )
    counts = np.asarray(hb.get_array(), dtype=float)
    hb.set_array(np.log1p(np.maximum(counts, 0.0)))
    hb.set_clim(0.0, HEX_VMAX)
    try:
        hb.set_rasterized(True)
    except Exception:
        pass

    _draw_reference_markers(ax_hex,
                            best_xy=_norm_env(*best_raw_xy, xlim),
                            chi0_xy=_norm_env(*chi0_raw_xy, xlim))

    ax_hex.set_xlim(0.0, 1.0); ax_hex.set_ylim(0.0, 1.0)
    ax_hex.set_aspect('equal', adjustable='box')

    x_tick_vals = np.arange(np.ceil(xlim[0]), np.floor(xlim[1]) + 1e-9, 1.0)
    xt_n    = (x_tick_vals - xlim[0]) / x_span_env
    xt_mask = (xt_n >= 0.0) & (xt_n <= 1.0)
    yt_n    = (CHI_YTICKS - GLOBAL_YLIM[0]) / y_span
    yt_mask = (yt_n >= 0.0) & (yt_n <= 1.0)

    ax_hex.set_xticks(xt_n[xt_mask])
    ax_hex.set_yticks(yt_n[yt_mask])
    ax_hex.set_xticklabels([f'{v:.0f}' for v in x_tick_vals[xt_mask]], fontsize=8)
    ax_hex.set_yticklabels([f'{v:.1f}' for v in CHI_YTICKS[yt_mask]], fontsize=8)
    ax_hex.set_xlabel(LABEL_FREE, fontsize=9)
    ax_hex.set_ylabel(LABEL_CHI,  fontsize=9, labelpad=8)
    ax_hex.set_title(f'{env_id}', fontsize=10, fontweight='bold')
    ax_hex.grid(alpha=0.22)

    legend_handles = [
        _reference_marker_legend_handle('*'),
        _reference_marker_legend_handle('o', markersize=5.5, dot_markersize=1.8),
    ]
    ax_hex.legend(legend_handles, [r'best $\sigma$ (GA)', r'$\chi = 0$'],
                  handler_map={tuple: HandlerTuple(ndivide=None)},
                  fontsize=7, loc='upper left', framealpha=0.75)

    cbar_hex = fig.colorbar(hb, cax=cax_hex)
    cbar_hex.set_ticks(HEX_TICK_POS.tolist())
    cbar_hex.set_ticklabels(HEX_TICK_LABELS)
    _style_colorbar_for_export(cbar_hex)
    cax_hex.yaxis.set_ticks_position('right')
    cax_hex.tick_params(labelsize=7)

    # ===== ΔF heatmap + marginal quiver =====
    # ΔF = F_best − F_baseline  (positive → relabelling raised free energy)
    if delta_free is not None:
        im_df = _plot_heatmap_panel(
            ax_df, env, delta_free,
            title=rf'$\Delta\mathcal{{F}}$ (best $\sigma$)',
            cmap=DELTA_FREE_CMAP,
            clim=DELTA_FREE_CLIM,
            show_goal=False,
            show_state_labels=False,
        )
        _annotate_heatmap(ax_df, delta_free, env,
                          cmap=DELTA_FREE_CMAP, clim=DELTA_FREE_CLIM)
        if marginal_policy is not None:
            disp.plot_quiver(env, marginal_policy, ax=ax_df, skip_walls=True)

        ax_df.text(0.5, -0.04, f'$\\sigma$ = {best_hash}',
                   ha='center', va='top', fontsize=5.5, color='#888888',
                   transform=ax_df.transAxes)
        if env_id == 'wrap_grid':
            _apply_dashed_border(ax_df)

        cbar_df = fig.colorbar(im_df, cax=cax_df)
        _style_colorbar_for_export(cbar_df)
        cax_df.yaxis.set_ticks_position('right')
        cax_df.tick_params(labelsize=7)
    else:
        ax_df.set_visible(False); cax_df.set_visible(False)

    # ===== ΔH heatmap + marginal quiver =====
    # ΔH = H_baseline − H_best  (green = goals agree more with best sigma)
    if delta_entropy is not None:
        im_ent = _plot_heatmap_panel(
            ax_ent, env, delta_entropy,
            title=r'$\Delta H[\pi_G]$ (bits)',
            cmap=DELTA_ENT_CMAP,
            clim=DELTA_ENT_CLIM,
            show_goal=False,
            show_state_labels=False,
        )
        _annotate_heatmap(ax_ent, delta_entropy, env,
                          cmap=DELTA_ENT_CMAP, clim=DELTA_ENT_CLIM)
        if marginal_policy is not None:
            disp.plot_quiver(env, marginal_policy, ax=ax_ent, skip_walls=True)

        ax_ent.text(0.5, -0.04, run_id,
                    ha='center', va='top', fontsize=5.5, color='#888888',
                    transform=ax_ent.transAxes)

        if env_id == 'wrap_grid':
            _apply_dashed_border(ax_ent)

        cbar_ent = fig.colorbar(im_ent, cax=cax_ent)
        _style_colorbar_for_export(cbar_ent)
        cax_ent.yaxis.set_ticks_position('right')
        cax_ent.tick_params(labelsize=7)
    else:
        ax_ent.set_visible(False); cax_ent.set_visible(False)

    for fmt in ('png', 'pdf'):
        fig.savefig(str(out_base) + f'.{fmt}', dpi=180, bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    print(f'  Saved → {out_base}.png')


print('Plot function defined.')

# ---------------------------------------------------------------------------
# SAVED — mean DI heatmap code (replace ax_df block above to use)
# ---------------------------------------------------------------------------
# mean_di = _load_mean_di(best_hive)   # add _load_mean_di to data loading cell
#
# def _load_mean_di(hive_dir):
#     arrays = []
#     for gf in sorted(hive_dir.glob('goal-*.npz'), key=lambda p: int(p.stem.split('-')[1])):
#         d = np.load(gf)
#         if 'info' in d.files:
#             arrays.append(np.asarray(d['info'], dtype=float))
#     return np.stack(arrays, axis=0).mean(axis=0) if arrays else None
#
# im = _plot_heatmap_panel(ax, env, mean_di,
#     title=rf'$\overline{{\mathcal{{I}}_D}}$ (best $\sigma$)',
#     cmap=DI_CMAP, clim=(DI_VMIN, DI_VMAX), show_goal=False, show_state_labels=False)
# _annotate_heatmap(ax, mean_di, env, cmap=DI_CMAP, clim=(DI_VMIN, DI_VMAX))
# cbar.set_ticks([0, 5, 10, 15])
# ---------------------------------------------------------------------------


In [ ]:
# ---------------------------------------------------------------------------
# Generate all panels
# ---------------------------------------------------------------------------

for env_id, d in env_data.items():
    plot_env_panel(env_id, d, FIGURES_DIR / f'figure5_{env_id}')

In [ ]:
# ---------------------------------------------------------------------------
# Inline display
# ---------------------------------------------------------------------------

from IPython.display import Image, display

for env_id in env_data:
    print(f'\n{env_id}')
    display(Image(filename=str(FIGURES_DIR / f'figure5_{env_id}.png')))

---

## Figure 5 — Paper version

Combined 2-row figure for publication.  Row 1: hexbin population scatter for each environment.
Row 2: $\Delta\mathcal{F}(s)$ heatmap for each environment.  Single shared colorbar per row.

**Configure here** — change `PAPER_ENV_ORDER` to reorder or swap environments;
add entries to `SIGMA_OVERRIDES` (env_id → sigma hash) to pin a different sigma for any environment.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — edit here to change environments or sigma selection
# ---------------------------------------------------------------------------

# Ordered list of environments to include (left → right).
PAPER_ENV_ORDER: list[str] = ['four_rooms', 'pillar_3', 'wrap_grid']
PAPER_ENV_ORDER: list[str] = ['four_rooms', 'wrap_grid']

# Optional sigma overrides: env_id → sigma hash.
# Leave empty to use each environment's best_sigma_hash from the GA run.
# Example: SIGMA_OVERRIDES = {'four_rooms': '24f551168e0469f95eea9fce2d976824'}
SIGMA_OVERRIDES: dict[str, str] = {}

PAPER_OUT = FIGURES_DIR / 'figure5_envs'

# ---------------------------------------------------------------------------
# Helper: load per-state data for a given sigma (override or cached best)
# ---------------------------------------------------------------------------

def _get_paper_data(env_id: str, sigma_hash: str | None = None) -> dict:
    """Return delta_free, marginal_policy for env_id, optionally with a pinned sigma."""
    d = env_data[env_id]

    if sigma_hash is None or sigma_hash == d['best_hash']:
        return {'delta_free': d['delta_free'], 'marginal_policy': d['marginal_policy'],
                'sigma_hash': d['best_hash']}

    # Reload data from the overridden hive dir
    run_dir  = d['run_dir']
    hive_dir = find_sigma_hive_dir(run_dir, sigma_hash)
    assert hive_dir is not None and hive_dir.exists(), \
        f'hive dir not found for sigma {sigma_hash} in {run_dir}'

    marginal, _  = load_goal_marginal_policy(hive_dir)
    mean_free    = _load_mean_free(hive_dir)

    agg       = d['agg']
    chi0_hash = str(agg.loc[agg['chi_twist'].idxmin(), 'sigma_hash'])
    shape     = d['shape']
    baseline  = _load_or_compute_baseline(run_dir, chi0_hash, env_id, shape, determinism=0.97)

    delta_free = mean_free - baseline['mean_free']
    return {'delta_free': delta_free, 'marginal_policy': marginal, 'sigma_hash': sigma_hash}


# ---------------------------------------------------------------------------
# Layout constants
# ---------------------------------------------------------------------------

_CELL  = 0.32
_N     = len(PAPER_ENV_ORDER)
_shape = env_data[PAPER_ENV_ORDER[0]]['shape']
_pw    = _shape[1] * _CELL
_ph    = _shape[0] * _CELL

_pad_l    = 0.72
_pad_r    = 0.05
_pad_b    = 0.55   # room for hexbin x-axis labels (hexbins are now bottom row)
_pad_b    = 0.05
_pad_t    = 0.12
_col_gap  = 0.16
_row_gap  = 0.16
_cbar_gap = 0.08
_cbar_w   = 0.08   # narrow, consistent with size='2%' used in individual panels

_fig_w = _pad_l + _N * _pw + (_N - 1) * _col_gap + _cbar_gap + _cbar_w + _pad_r
_fig_h = _pad_t + _ph + _row_gap + _ph + _pad_b

# ΔF row on top, hexbin row on bottom
_y_hex = _pad_b / _fig_h                            # bottom row — hexbins
_y_df  = (_pad_b + _ph + _row_gap) / _fig_h         # top row  — ΔF heatmaps

_x_cols = [(_pad_l + i * (_pw + _col_gap)) / _fig_w for i in range(_N)]
_x_cbar = (_pad_l + _N * _pw + (_N - 1) * _col_gap + _cbar_gap) / _fig_w

print(f'Figure size : {_fig_w:.2f} × {_fig_h:.2f} inches')
print(f'Panel size  : {_pw:.2f} × {_ph:.2f} inches')

# ---------------------------------------------------------------------------
# Draw
# ---------------------------------------------------------------------------
import matplotlib.patheffects as _pe

fig = plt.figure(figsize=(_fig_w, _fig_h))

ax_hex  = [fig.add_axes([_x_cols[i], _y_hex, _pw / _fig_w, _ph / _fig_h]) for i in range(_N)]
ax_df   = [fig.add_axes([_x_cols[i], _y_df,  _pw / _fig_w, _ph / _fig_h]) for i in range(_N)]
cax_hex = fig.add_axes([_x_cbar, _y_hex, _cbar_w / _fig_w, _ph / _fig_h])
cax_df  = fig.add_axes([_x_cbar, _y_df,  _cbar_w / _fig_w, _ph / _fig_h])

_last_hb = None
_last_im = None

for col, env_id in enumerate(PAPER_ENV_ORDER):
    d        = env_data[env_id]
    agg      = d['agg']
    xlim     = d['xlim']
    x_span   = xlim[1] - xlim[0]
    env      = d['env']

    sigma_hash = SIGMA_OVERRIDES.get(env_id)
    paper_data = _get_paper_data(env_id, sigma_hash)
    delta_free      = paper_data['delta_free']
    marginal_policy = paper_data['marginal_policy']
    used_hash       = paper_data['sigma_hash']

    # ── ΔF heatmap (top row) ────────────────────────────────────────────────
    im = _plot_heatmap_panel(
        ax_df[col], env, delta_free,
        title='',
        cmap=DELTA_FREE_CMAP,
        clim=DELTA_FREE_CLIM,
        show_goal=False,
        show_state_labels=False,
    )
    _annotate_heatmap(ax_df[col], delta_free, env,
                      cmap=DELTA_FREE_CMAP, clim=DELTA_FREE_CLIM, size=4.5)
    if marginal_policy is not None:
        disp.plot_quiver(env, marginal_policy, ax=ax_df[col], skip_walls=True)
    _last_im = im

    """ sigma and run id annotation
    run_id = d['run_dir'].name
    ax_df[col].text(
        0.5, -0.03, f'$\\sigma$ = {used_hash}',
        ha='center', va='top', fontsize=4.5, color='#999999',
        transform=ax_df[col].transAxes,
    )
    ax_df[col].text(
        0.5, -0.07, run_id,
        ha='center', va='top', fontsize=4.0, color='#bbbbbb',
        transform=ax_df[col].transAxes,
    )
    """
    if env_id == 'wrap_grid':
        _apply_dashed_border(ax_df[col])

    # ── hexbin (bottom row) ──────────────────────────────────────────────────
    x_raw = agg['mean_free'].values.astype(float)
    y_raw = agg['chi_twist'].values.astype(float)
    x_n, y_n = _norm_env(x_raw, y_raw, xlim)

    hb = ax_hex[col].hexbin(
        x_n, y_n, gridsize=HEX_GRIDSIZE, mincnt=1,
        extent=[0., 1., 0., 1.], cmap='Purples', linewidths=0., zorder=2,
    )
    counts = np.asarray(hb.get_array(), dtype=float)
    hb.set_array(np.log1p(np.maximum(counts, 0.)))
    hb.set_clim(0., HEX_VMAX)
    try:
        hb.set_rasterized(True)
    except Exception:
        pass
    _last_hb = hb

    # Reference markers — unfilled, fine lines, white halo for hexbin visibility
    bx, by = _norm_env(*d['best_raw_xy'], xlim)
    cx, cy = _norm_env(*d['chi0_raw_xy'], xlim)
    _halo = [_pe.withStroke(linewidth=3.0, foreground='white')]

    # Star: unfilled, fine lineweight
    ax_hex[col].plot(
        float(bx), float(by),
        marker='*', markersize=10, markerfacecolor='none',
        markeredgecolor='#111111', markeredgewidth=0.8,
        linestyle='none', zorder=11,
        path_effects=_halo,
    )
    # Circle: unfilled, fine lineweight
    ax_hex[col].plot(
        float(cx), float(cy),
        marker='o', markersize=6, markerfacecolor='none',
        markeredgecolor='#111111', markeredgewidth=1.1,
        linestyle='none', zorder=11,
        path_effects=_halo,
    )

    ax_hex[col].set_xlim(0., 1.); ax_hex[col].set_ylim(0., 1.)
    ax_hex[col].set_aspect('equal', adjustable='box')

    xt_vals = np.arange(np.ceil(xlim[0]), np.floor(xlim[1]) + 1e-9, 1.)
    xt_n    = (xt_vals - xlim[0]) / x_span
    xt_mask = (xt_n >= 0.) & (xt_n <= 1.)
    yt_n    = (CHI_YTICKS - GLOBAL_YLIM[0]) / y_span
    yt_mask = (yt_n >= 0.) & (yt_n <= 1.)

    ax_hex[col].set_xticks(xt_n[xt_mask])
    ax_hex[col].set_xticklabels([f'{v:.0f}' for v in xt_vals[xt_mask]], fontsize=7)
    ax_hex[col].set_xlabel(
        r'$E_{s,g}\!\left[\,\mathcal{F}^{\pi}_{\beta}(s)\,\right]$', fontsize=8
    )
    ax_hex[col].grid(alpha=0.22)

    if col == 0:
        ax_hex[col].set_yticks(yt_n[yt_mask])
        ax_hex[col].set_yticklabels([f'{v:.1f}' for v in CHI_YTICKS[yt_mask]], fontsize=7)
        ax_hex[col].set_ylabel(LABEL_CHI, fontsize=8, labelpad=6)
        ax_hex[col].legend(
            [
                ax_hex[col].plot([], [], marker='*', color='#111111', linestyle='none',
                                 markersize=7, markerfacecolor='none', markeredgewidth=0.8)[0],
                ax_hex[col].plot([], [], marker='o', color='#111111', linestyle='none',
                                 markersize=5, markerfacecolor='none', markeredgewidth=1.1)[0],
            ],
            [r'best $\sigma$', r'$\chi = 0$'],
            handler_map={tuple: HandlerTuple(ndivide=None)},
            fontsize=6.5, loc='upper left', framealpha=0.75,
        )
    else:
        ax_hex[col].set_yticks(yt_n[yt_mask])  # keep ticks for grid lines
        ax_hex[col].set_yticklabels([])     # but hide labels

# ── shared colorbars ─────────────────────────────────────────────────────────

cbar_hex = fig.colorbar(_last_hb, cax=cax_hex)
cbar_hex.set_ticks(HEX_TICK_POS.tolist())
cbar_hex.set_ticklabels(HEX_TICK_LABELS)
_style_colorbar_for_export(cbar_hex)
cax_hex.yaxis.set_ticks_position('right')
cax_hex.tick_params(labelsize=6.5)
cbar_hex.set_label('hex occupancy', fontsize=7, labelpad=4)

cbar_df = fig.colorbar(_last_im, cax=cax_df)
_style_colorbar_for_export(cbar_df)
cax_df.yaxis.set_ticks_position('right')
cax_df.tick_params(labelsize=6.5)
# E_g[...] makes explicit that ΔF is per-state (s fixed), averaged over goals only
# (contrast with hexbin x-axis E_{s,g}[...] which also averages over states)
cbar_df.set_label(
    r'$\Delta\bar{\mathcal{F}}(s) = E_g\!\left[\mathcal{F}^*_{\sigma^\star}(s,g)\right]'
    r' - E_g\!\left[\mathcal{F}^*_\mathrm{base}(s,g)\right]$',
    fontsize=7, labelpad=4,
)

# ── save ─────────────────────────────────────────────────────────────────────
for fmt in ('png', 'pdf'):
    fig.savefig(str(PAPER_OUT) + f'.{fmt}', dpi=200, bbox_inches='tight', pad_inches=0.03)
plt.close(fig)
print(f'Saved → {PAPER_OUT}.png')

from IPython.display import Image, display
display(Image(filename=str(PAPER_OUT) + '.png'))


---

## Diagnostic figure: ΔF decomposition into ΔI and ΔV

Shows, for each environment, how the GA-best twist redistributes planning cost
into its two components:

- **Baseline DI** $\bar{\mathcal{I}}_D^{\mathrm{base}}(s)$ with baseline quiver
- **Best DI** $\bar{\mathcal{I}}_D^{\sigma^\star}(s)$ with best quiver
- **ΔI** = best DI − baseline DI  (positive = twist raised informational cost)
- **ΔV** = best value − baseline value  (positive = twist improved value)
- **ΔF** (existing; = −ΔV + ΔI at β=1)

This lets us see whether ΔF ≈ 0 at some states because the twist is neutral there,
or because ΔI and ΔV partially cancel.


In [ ]:
# ---------------------------------------------------------------------------
# Diagnostic figure: ΔF decomposition into ΔI, ΔV, ΔH — for each environment
# ---------------------------------------------------------------------------
# Colour conventions (consistent with paper figures):
#   DI (absolute)        → 'Oranges'       (DI_CMAP)
#   Entropy (absolute)   → 'Blues'          (darker = more uncertain, "royal blue")
#   ΔI                   → 'PuOr'           (orange = DI ↑ / worse; purple = DI ↓ / better)
#   ΔH                   → 'RdYlGn'         (green = entropy ↓ / goals agree more; red = entropy ↑)
#   ΔV                   → 'RdBu'           (blue = value improved; red = value worsened)
#   ΔF                   → DELTA_FREE_CMAP  (purple = cost reduced; green = cost increased)
# ---------------------------------------------------------------------------

_DIAG_ENV_ORDER = PAPER_ENV_ORDER
_DIAG_OUT       = FIGURES_DIR / 'figure5_diagnostic_decomposition'

# ── shared colour scales ──────────────────────────────────────────────────────

_all_abs_di = np.concatenate([
    d['best_mean_info'][np.isfinite(d['best_mean_info'])]
    for d in env_data.values() if d.get('best_mean_info') is not None
] + [
    d['baseline_mean_info'][np.isfinite(d['baseline_mean_info'])]
    for d in env_data.values() if d.get('baseline_mean_info') is not None
])
ABS_DI_CLIM = (0.0, float(np.percentile(_all_abs_di, 99)))

# Entropy: max is log2(4) = 2 bits for 4-action uniform policy
ABS_ENT_CLIM = (0.0, np.log2(4))

_all_di = np.concatenate([
    d['delta_info'][np.isfinite(d['delta_info'])]
    for d in env_data.values() if d.get('delta_info') is not None
])
_di_abs = float(max(abs(float(_all_di.min())), abs(float(_all_di.max()))))
DELTA_INFO_CLIM = (-_di_abs, _di_abs)

_all_dh = np.concatenate([
    d['delta_entropy'][np.isfinite(d['delta_entropy'])]
    for d in env_data.values() if d.get('delta_entropy') is not None
])
_dh_abs = float(max(abs(float(_all_dh.min())), abs(float(_all_dh.max()))))
DELTA_ENT2_CLIM = (-_dh_abs, _dh_abs)

_all_dv = np.concatenate([
    d['delta_value'][np.isfinite(d['delta_value'])]
    for d in env_data.values() if d.get('delta_value') is not None
])
_dv_abs = float(max(abs(float(_all_dv.min())), abs(float(_all_dv.max()))))
DELTA_VAL_CLIM = (-_dv_abs, _dv_abs)

print(f'Absolute DI  clim : {ABS_DI_CLIM}')
print(f'Absolute Ent clim : {ABS_ENT_CLIM}')
print(f'ΔI           clim : {DELTA_INFO_CLIM}')
print(f'ΔH           clim : {DELTA_ENT2_CLIM}')
print(f'ΔV           clim : {DELTA_VAL_CLIM}')
print(f'ΔF           clim : {DELTA_FREE_CLIM}')

# ── row definitions ───────────────────────────────────────────────────────────
# (data_key, quiver_key, cmap, clim, row_label, cbar_label)
_ROWS = [
    ('baseline_mean_info', 'baseline_marginal', DI_CMAP,         ABS_DI_CLIM,
     r'Baseline $\bar{\mathcal{I}}_D(s)$',
     r'$\bar{\mathcal{I}}_D$ (nats)'),

    ('best_mean_info',     'marginal_policy',   DI_CMAP,         ABS_DI_CLIM,
     r'Best $\bar{\mathcal{I}}_D(s)$',
     r'$\bar{\mathcal{I}}_D$ (nats)'),

    ('delta_info',         'marginal_policy',   'PuOr',          DELTA_INFO_CLIM,
     r'$\Delta\mathcal{I}_D$ (best $-$ base)',
     r'orange = DI ↑ (worse) | purple = DI ↓ (better)'),

    ('baseline_entropy',   'baseline_marginal', 'Blues',         ABS_ENT_CLIM,
     r'Baseline $H[\pi_G](s)$ (bits)',
     r'entropy (bits);  max = $\log_2 4 = 2$'),

    ('best_entropy',       'marginal_policy',   'Blues',         ABS_ENT_CLIM,
     r'Best $H[\pi_G](s)$ (bits)',
     r'entropy (bits);  max = $\log_2 4 = 2$'),

    ('delta_entropy',      'marginal_policy',   DELTA_ENT_CMAP,  DELTA_ENT2_CLIM,
     r'$\Delta H[\pi_G]$ (base $-$ best, bits)',
     r'green = entropy ↓ (goals agree more) | red = entropy ↑'),

    ('delta_value',        'marginal_policy',   'RdBu',          DELTA_VAL_CLIM,
     r'$\Delta V$ (best $-$ base)',
     r'blue = value improved | red = value worsened'),

    ('delta_free',         'marginal_policy',   DELTA_FREE_CMAP, DELTA_FREE_CLIM,
     r'$\Delta\mathcal{F}$ (best $-$ base)',
     r'purple = cost reduced | green = cost increased'),
]
_NROWS = len(_ROWS)

# ── layout ────────────────────────────────────────────────────────────────────
_N     = len(_DIAG_ENV_ORDER)
_shape = env_data[_DIAG_ENV_ORDER[0]]['shape']
_CELL  = 0.34
_pw    = _shape[1] * _CELL
_ph    = _shape[0] * _CELL

_pad_l    = 1.85
_pad_r    = 0.05
_pad_b    = 0.30
_pad_t    = 0.40
_col_gap  = 0.28
_row_gap  = 0.36
_cbar_gap = 0.14
_cbar_w   = 0.10

_fig_w = _pad_l + _N * _pw + (_N - 1) * _col_gap + _cbar_gap + _cbar_w + _pad_r
_fig_h = _pad_t + _NROWS * _ph + (_NROWS - 1) * _row_gap + _pad_b

_y_rows = [
    (_pad_b + (_NROWS - 1 - r) * (_ph + _row_gap)) / _fig_h
    for r in range(_NROWS)
]
_x_cols = [(_pad_l + i * (_pw + _col_gap)) / _fig_w for i in range(_N)]
_x_cbar = (_pad_l + _N * _pw + (_N - 1) * _col_gap + _cbar_gap) / _fig_w

print(f'\nFigure size: {_fig_w:.2f} × {_fig_h:.2f} inches')

fig = plt.figure(figsize=(_fig_w, _fig_h))

axes  = [[fig.add_axes([_x_cols[c], _y_rows[r], _pw / _fig_w, _ph / _fig_h])
          for c in range(_N)]
         for r in range(_NROWS)]
caxes = [fig.add_axes([_x_cbar, _y_rows[r], _cbar_w / _fig_w, _ph / _fig_h])
         for r in range(_NROWS)]

_last_ims = [None] * _NROWS

for col, env_id in enumerate(_DIAG_ENV_ORDER):
    d   = env_data[env_id]
    env = d['env']

    for row, (dkey, qkey, cmap, clim, row_label, cbar_label) in enumerate(_ROWS):
        ax   = axes[row][col]
        data = d.get(dkey)
        quiv = d.get(qkey)

        if data is None:
            ax.set_visible(False)
            continue

        col_title = env_id.replace('_', ' ') if row == 0 else ''
        im = _plot_heatmap_panel(
            ax, env, data,
            title=col_title,
            cmap=cmap, clim=clim,
            show_goal=False, show_state_labels=False,
        )
        _annotate_heatmap(ax, data, env, cmap=cmap, clim=clim, size=4.0)

        if quiv is not None:
            disp.plot_quiver(env, quiv, ax=ax, skip_walls=True)

        if env_id == 'wrap_grid':
            _apply_dashed_border(ax)

        if col == 0:
            ax.set_ylabel(row_label, fontsize=7, labelpad=6)

        _last_ims[row] = im

# Shared colorbars
for row, (_, _, _, _, _, cbar_label) in enumerate(_ROWS):
    if _last_ims[row] is None:
        caxes[row].set_visible(False)
        continue
    cbar = fig.colorbar(_last_ims[row], cax=caxes[row])
    _style_colorbar_for_export(cbar)
    caxes[row].yaxis.set_ticks_position('right')
    caxes[row].tick_params(labelsize=5.5)
    cbar.set_label(cbar_label, fontsize=5.5, labelpad=3)

for fmt in ('png', 'pdf'):
    fig.savefig(str(_DIAG_OUT) + f'.{fmt}', dpi=180, bbox_inches='tight', pad_inches=0.04)
plt.close(fig)
print(f'\nSaved → {_DIAG_OUT}.png')

from IPython.display import Image, display
display(Image(filename=str(_DIAG_OUT) + '.png'))


## Alternative: 2-environment figure (no pillar)

Re-render the paper composite with just four_rooms and wrap_grid.
Change `PAPER_ENV_ORDER` above to `['four_rooms', 'wrap_grid']` and
re-run the composite cell, or use the cell below which builds it directly.

In [ ]:
# --- Alternative figure 5: two environments only (four_rooms + wrap_grid) ---
# Re-uses env_data and plot functions already computed above.
# Just rebuilds the composite with N=2.

ALT_ENV_ORDER = ['four_rooms', 'wrap_grid']
_N_alt = len(ALT_ENV_ORDER)
ALT_OUT = FIGURES_DIR / 'figure5_envs_alt_2env'

# Reuse the same composite-building code but with 2 columns
# This requires the composite cell (cell 12) to have been run first.
# We just re-call the plot function with the reduced env list.

try:
    # check that the plotting function from cell 12 is available
    _shape_alt = env_data[ALT_ENV_ORDER[0]]['shape']
    _CELL_alt = 0.34
    _pw_alt = _shape_alt[1] * _CELL_alt
    _ph_alt = _shape_alt[0] * _CELL_alt

    _pad_l = 0.72; _pad_r = 0.15; _pad_t = 0.12; _pad_b = 0.60
    _col_gap = 0.55; _row_gap = 0.46
    _cbar_w = 0.12; _cbar_gap = 0.14

    _fig_w_alt = _pad_l + _N_alt * _pw_alt + (_N_alt - 1) * _col_gap + _cbar_gap + _cbar_w + _pad_r
    _fig_h_alt = _pad_t + _ph_alt + _row_gap + _ph_alt + _pad_b

    print(f'Alternative figure 5: {ALT_ENV_ORDER}')
    print(f'Figure size: {_fig_w_alt:.1f} x {_fig_h_alt:.1f} inches')
    print(f'To generate: change PAPER_ENV_ORDER to {ALT_ENV_ORDER} in the composite cell above and re-run.')
    print(f'Output will be saved to {ALT_OUT}')

except NameError:
    print('Run the composite cell (cell 12) first, then re-run this cell.')

---

## Per-action habit cycles for the four-rooms paper sigma

For each action label, build the deterministic per-label dynamics
(`det = 1` skeleton with `goals = []`) and identify the cycles + basins.
This is the same diagnostic we use for the wrap_grid review response,
applied to the paper four_rooms sigma. Useful for verifying the
"habit cycle" claim in Section 4.1 — if the GA-best four-rooms sigma
has a label-N cycle near goal 45, this is where we'd see it.


In [ ]:
# ---------------------------------------------------------------------------
# Per-action habit cycle analysis for the four_rooms paper sigma
# ---------------------------------------------------------------------------
from gridcore.envs import GridRoom
from gridcore.planning.state_distribution import (
    get_stationary_distribution_eigen_decomposition,
)
from gridvis.display_twist import (
    _action_color_indices,
    _action_labels,
    _select_action_colours,
)
from figure_support import _walls_for_env
from figure_support import RUN_DIR as FOUR_ROOMS_RUN_DIR, BEST_SIGMA_HASH

# Build a goal-free four_rooms env so the per-label dynamics are not
# contaminated by the goal-absorbing transition.
_walls_fr = _walls_for_env('four_rooms', (7, 7))
_env_fr = GridRoom({
    'shape': (7, 7),
    'goals': [],
    'manhattan': True,
    'determinism': 0.97,
    'epsilon': 0.0,
    'twist_seed': 0,
    'walls': _walls_fr,
})

# Load the paper sigma and apply it
_summary_fr = json.loads(
    next(FOUR_ROOMS_RUN_DIR.glob('*.summary.json')).read_text()
)
_sigma_path_fr = next(
    FOUR_ROOMS_RUN_DIR.rglob(f'sigma_id={BEST_SIGMA_HASH}/sigma.npy')
)
_sigma_fr = np.load(_sigma_path_fr)

_available_fr = np.asarray(_env_fr.available_states, dtype=int)
_env_fr.sigma[_available_fr] = _sigma_fr[_available_fr]
_env_fr.twist_dynamics()
_env_fr.update_dynamics_for_goals(_env_fr.goals)

print(f'four_rooms sigma: {BEST_SIGMA_HASH[:16]}\u2026')
print(f'env: shape={_env_fr.shape}, nS={_env_fr.nS}, nA={_env_fr.nA}, walls={len(_walls_fr) if _walls_fr else 0}')


# Functional graph decomposition (vendored from review/run_comparison)
def _deterministic_successor(env, action):
    T = np.asarray(env.T)
    succ = np.zeros(env.nS, dtype=int)
    for s in range(env.nS):
        succ[s] = int(np.argmax(T[s * env.nA + action]))
    return succ


def _find_basins(succ):
    n = len(succ)
    visited = np.full(n, -1, dtype=int)
    bid_arr = np.full(n, -1, dtype=int)
    cycles = []
    cb = 0
    for start in range(n):
        if visited[start] >= 0:
            continue
        path = []
        s = start
        while visited[s] < 0:
            visited[s] = start
            path.append(s)
            s = int(succ[s])
        if visited[s] == start:
            ci = path.index(s)
            cycles.append(path[ci:])
            for p in path:
                bid_arr[p] = cb
            cb += 1
        else:
            existing = bid_arr[s]
            for p in path:
                bid_arr[p] = existing
    return {
        'cycles': cycles,
        'basin_id': bid_arr,
        'basin_sizes': [int(np.sum(bid_arr == b)) for b in range(cb)],
    }


_action_lbls_fr = _action_labels(_env_fr)
_per_label_fr = []
for a in range(_env_fr.nA):
    succ = _deterministic_successor(_env_fr, a)
    decomp = _find_basins(succ)
    pi = np.zeros((_env_fr.nS, _env_fr.nA))
    pi[:, a] = 1.0
    d = get_stationary_distribution_eigen_decomposition(_env_fr, pi)

    cycle_state_set = set()
    cycle_infos = []
    for ci, cycle in enumerate(decomp['cycles']):
        cstates = [int(s) for s in cycle]
        cycle_infos.append({
            'states': cstates,
            'basin_size': int(decomp['basin_sizes'][ci]),
            'mass': float(sum(d[s] for s in cstates)),
        })
        cycle_state_set.update(cstates)
    _per_label_fr.append({
        'label': _action_lbls_fr[a],
        'action_idx': a,
        'cycles': cycle_infos,
        'basin_id': decomp['basin_id'],
        'succ': succ,
        'stationary': d,
        'mass_on_cycles': float(sum(d[s] for s in cycle_state_set)),
    })

print()
print(f'{"label":>5s}  {"n cycles":>9s}  {"max basin":>10s}  {"mass on cycles":>15s}')
for info in _per_label_fr:
    max_basin = max(c['basin_size'] for c in info['cycles']) if info['cycles'] else 0
    print(f"{info['label']:>5s}  {len(info['cycles']):>9d}  {max_basin:>10d}  {info['mass_on_cycles']:>15.3f}")


In [ ]:
# ---------------------------------------------------------------------------
# 2x2 basin decomposition plot (functional graph) for four_rooms
# ---------------------------------------------------------------------------
from matplotlib.patches import Rectangle as _Rect

_h_fr, _w_fr = _env_fr.shape
_color_ids_fr, _palette_size_fr = _action_color_indices(_action_lbls_fr)
_action_colours_fr = _select_action_colours(_palette_size_fr)
_state_ids_fr = np.arange(_env_fr.nS).reshape(_h_fr, _w_fr)
_walls_set_fr = set(int(s) for s in getattr(_env_fr, 'walls_flat', []))

fig_b_fr, axes_b_fr = plt.subplots(2, 2, figsize=(8.5, 8.5))
basin_cmap_fr = plt.cm.Set2

for idx, info in enumerate(_per_label_fr):
    ax = axes_b_fr[idx // 2, idx % 2]
    a = info['action_idx']
    succ = info['succ']
    bid = info['basin_id']
    n_basins = len(info['cycles'])

    grid = bid.reshape(_h_fr, _w_fr).astype(float)
    mask = np.zeros((_h_fr, _w_fr), dtype=bool)
    for s in _walls_set_fr:
        mask[s // _w_fr, s % _w_fr] = True
    grid_masked = nma.array(grid, mask=mask)
    ax.imshow(
        grid_masked, cmap=basin_cmap_fr,
        vmin=-0.5, vmax=max(n_basins - 0.5, 0.5),
        origin='upper', interpolation='nearest',
    )

    cycle_states = set()
    for c in info['cycles']:
        cycle_states.update(c['states'])
    label_colour = _action_colours_fr[_color_ids_fr[a]]

    for r in range(_h_fr):
        for c in range(_w_fr):
            s = r * _w_fr + c
            if s in _walls_set_fr:
                ax.add_patch(_Rect(
                    (c - 0.5, r - 0.5), 1.0, 1.0,
                    facecolor='#bbbbbb', edgecolor='none', zorder=1,
                ))
                continue
            if s in cycle_states:
                ax.add_patch(_Rect(
                    (c - 0.5, r - 0.5), 1.0, 1.0,
                    facecolor='none', edgecolor=label_colour,
                    linewidth=0.0, hatch='///', alpha=0.7, zorder=2,
                ))
            s_next = int(succ[s])
            r2, c2 = divmod(s_next, _w_fr)
            dr, dc = r2 - r, c2 - c
            if abs(dr) > 1: dr = -np.sign(dr)
            if abs(dc) > 1: dc = -np.sign(dc)
            if dr == 0 and dc == 0:
                ax.plot(c, r, 'o', color='red', markersize=4, zorder=4)
            else:
                ax.annotate(
                    '', xy=(c + dc * 0.4, r + dr * 0.4),
                    xytext=(c - dc * 0.1, r - dr * 0.1),
                    arrowprops=dict(arrowstyle='->', color='black',
                                    lw=1.0, mutation_scale=8),
                    zorder=3,
                )

    disp.add_label_to_plot(
        ax, _state_ids_fr, shift=-0.35, size='6', env=_env_fr,
        skip_goals=False, format_fn=lambda v: str(int(v)),
    )

    cycle_descs = []
    for c in info['cycles']:
        cycle_str = '\u2192'.join(str(s) for s in c['states'])
        cycle_descs.append(
            f'[{cycle_str}] bsn {c["basin_size"]} mass {c["mass"]:.2f}'
        )
    ax.set_xlabel('\n'.join(cycle_descs), fontsize=7)
    ax.set_title(f'Label "{info["label"]}" \u2014 {n_basins} cycle(s)', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim(-0.5, _w_fr - 0.5)
    ax.set_ylim(_h_fr - 0.5, -0.5)

fig_b_fr.suptitle(
    f'Functional graph decomposition \u2014 four_rooms (sigma {BEST_SIGMA_HASH[:16]}\u2026)',
    fontsize=12, y=0.99,
)
fig_b_fr.subplots_adjust(hspace=0.42, wspace=0.15, top=0.93)
fig_b_fr.savefig(FIGURES_DIR / 'figure5_four_rooms_per_action_basins.png',
                 dpi=140, bbox_inches='tight')
fig_b_fr.savefig(FIGURES_DIR / 'figure5_four_rooms_per_action_basins.pdf',
                 bbox_inches='tight')
plt.show()
print('Saved: figures/figure5_four_rooms_per_action_basins.[png|pdf]')


In [ ]:
# ---------------------------------------------------------------------------
# 1x4 per-label stationary distribution plot for four_rooms
# ---------------------------------------------------------------------------
from matplotlib.gridspec import GridSpec

_vmax_fr = max(float(info['stationary'].max()) for info in _per_label_fr)

fig_s_fr = plt.figure(figsize=(13, 3.8))
gs_fr = GridSpec(1, 5, width_ratios=[1, 1, 1, 1, 0.04], wspace=0.15)
axes_s_fr = [fig_s_fr.add_subplot(gs_fr[0, i]) for i in range(4)]
cax_s_fr = fig_s_fr.add_subplot(gs_fr[0, 4])

im_fr = None
for idx, info in enumerate(_per_label_fr):
    ax = axes_s_fr[idx]
    d = info['stationary']
    grid = d.reshape(_h_fr, _w_fr)
    mask = np.zeros((_h_fr, _w_fr), dtype=bool)
    for s in _walls_set_fr:
        mask[s // _w_fr, s % _w_fr] = True
    grid_masked = nma.array(grid, mask=mask)
    im_fr = ax.imshow(
        grid_masked, cmap='Oranges', vmin=0, vmax=_vmax_fr,
        interpolation='nearest',
    )
    for r in range(_h_fr):
        for c in range(_w_fr):
            s = r * _w_fr + c
            if s in _walls_set_fr:
                ax.add_patch(_Rect(
                    (c - 0.5, r - 0.5), 1.0, 1.0,
                    facecolor='#bbbbbb', edgecolor='none', zorder=1,
                ))
                continue
            v = grid[r, c]
            if v > 0.005:
                ax.text(
                    c, r, f'{v:.2f}',
                    ha='center', va='center', fontsize=6,
                    color='white' if v > _vmax_fr * 0.5 else 'black',
                )
    ax.set_title(f'following label {info["label"]}', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

if im_fr is not None:
    cbar_fr = fig_s_fr.colorbar(im_fr, cax=cax_s_fr)
    cbar_fr.set_label(r'$d^a(s)$', fontsize=10)
fig_s_fr.suptitle(
    f'Per-label stationary $d^a(s)$ \u2014 four_rooms (sigma {BEST_SIGMA_HASH[:16]}\u2026)',
    fontsize=12, y=1.00,
)
fig_s_fr.savefig(FIGURES_DIR / 'figure5_four_rooms_per_action_stationary.png',
                 dpi=140, bbox_inches='tight')
fig_s_fr.savefig(FIGURES_DIR / 'figure5_four_rooms_per_action_stationary.pdf',
                 bbox_inches='tight')
plt.show()
print('Saved: figures/figure5_four_rooms_per_action_stationary.[png|pdf]')
